In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import json
import random

# --- 1. ENHANCED CONFIGURATION ---
CONFIG = {
    'data_dir': Path("/home/rbielski/SAL_Git_Projects/Cheatgrass/cheatgrass_data_quality_filtered"),
    'output_dir': Path("/home/rbielski/SAL_Git_Projects/Cheatgrass/model1_output"),
    'batch_size': 1,
    'learning_rate': 1e-4,
    'epochs': 50,
    'hidden_dim': 128,
    'input_bands': 6,
    'train_val_split_size': 0.2,
    'training_window_size': 32,
    'enable_augmentation': True,
    'validation_crops_per_sample': 3,  # Multiple random crops during validation too!
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}
CONFIG['output_dir'].mkdir(exist_ok=True)
print(f"Using device: {CONFIG['device']}")
print(f"Training window size: {CONFIG['training_window_size']}x{CONFIG['training_window_size']}")
print(f"FIXED: Validation will also use random cropping!")

# --- 2. ADVANCED LOSS FUNCTION (Focal Tversky) ---
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha, beta, gamma=1):
        super(FocalTverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, inputs, targets):
        # Apply sigmoid to convert logits to probabilities
        inputs = torch.sigmoid(inputs)
        
        tp = (inputs * targets).sum()
        fp = ((1 - targets) * inputs).sum()
        fn = (targets * (1 - inputs)).sum()

        tversky = (tp) / (tp + self.alpha * fp + self.beta * fn + 1e-6)
        return (1 - tversky) ** self.gamma

# --- 3. ENHANCED RANDOM CROPPING ---
def load_sample_metadata(data_dir, sample_id):
    """Load metadata for a sample to determine its context size."""
    metadata_path = data_dir / f"{sample_id}_metadata.json"
    if metadata_path.exists():
        try:
            with open(metadata_path, 'r') as f:
                return json.load(f)
        except:
            pass
    return None

def find_valid_crop_position_enhanced(mask, crop_size, force_include_vegetation=True, min_mask_pixels=1):
    """
    Enhanced crop position finding with better spatial distribution.
    
    Args:
        mask: Input mask of shape (H, W)
        crop_size: Size of the crop window
        force_include_vegetation: If True, ensures vegetation is included
        min_mask_pixels: Minimum number of mask pixels required in crop
    
    Returns:
        (top, left) coordinates for cropping
    """
    h, w = mask.shape
    if h <= crop_size or w <= crop_size:
        return 0, 0
    
    # Find all positions where mask has pixels
    mask_positions = np.where(mask > 0)
    
    if len(mask_positions[0]) == 0 or not force_include_vegetation:
        # No mask pixels or not forcing inclusion - return truly random position
        top = random.randint(0, h - crop_size)
        left = random.randint(0, w - crop_size)
        return top, left
    
    # Strategy: Encourage vegetation to appear in different parts of the crop window
    max_attempts = 100
    valid_positions = []
    
    for attempt in range(max_attempts):
        # Choose a random mask pixel as anchor
        idx = random.randint(0, len(mask_positions[0]) - 1)
        anchor_y, anchor_x = mask_positions[0][idx], mask_positions[1][idx]
        
        # Randomly position the anchor in different parts of the crop window
        # This is key: don't always center the vegetation!
        if random.random() < 0.2:  # 20% chance: vegetation in top-left
            anchor_offset_y = random.randint(0, crop_size // 4)
            anchor_offset_x = random.randint(0, crop_size // 4)
        elif random.random() < 0.4:  # 20% chance: vegetation in top-right
            anchor_offset_y = random.randint(0, crop_size // 4)
            anchor_offset_x = random.randint(3 * crop_size // 4, crop_size - 1)
        elif random.random() < 0.6:  # 20% chance: vegetation in bottom-left
            anchor_offset_y = random.randint(3 * crop_size // 4, crop_size - 1)
            anchor_offset_x = random.randint(0, crop_size // 4)
        elif random.random() < 0.8:  # 20% chance: vegetation in bottom-right
            anchor_offset_y = random.randint(3 * crop_size // 4, crop_size - 1)
            anchor_offset_x = random.randint(3 * crop_size // 4, crop_size - 1)
        else:  # 20% chance: vegetation anywhere (including center)
            anchor_offset_y = random.randint(0, crop_size - 1)
            anchor_offset_x = random.randint(0, crop_size - 1)
        
        top = anchor_y - anchor_offset_y
        left = anchor_x - anchor_offset_x
        
        # Ensure crop is within bounds
        top = max(0, min(top, h - crop_size))
        left = max(0, min(left, w - crop_size))
        
        # Check if this crop has enough mask pixels
        crop_mask = mask[top:top+crop_size, left:left+crop_size]
        if np.sum(crop_mask) >= min_mask_pixels:
            valid_positions.append((top, left))
            
            # Early return if we have a good position
            if len(valid_positions) >= 5:
                break
    
    if valid_positions:
        return random.choice(valid_positions)
    
    # Fallback: random position that includes some vegetation
    for _ in range(20):
        top = random.randint(0, h - crop_size)
        left = random.randint(0, w - crop_size)
        crop_mask = mask[top:top+crop_size, left:left+crop_size]
        if np.sum(crop_mask) >= min_mask_pixels:
            return top, left
    
    # Final fallback: center crop
    top = (h - crop_size) // 2
    left = (w - crop_size) // 2
    return top, left

def enhanced_random_crop_sample(data, mask, crop_size, is_training=True, force_spatial_diversity=True):
    """
    Enhanced random cropping with true spatial diversity.
    """
    t, c, h, w = data.shape
    
    if h == crop_size and w == crop_size:
        return data, mask
    
    # Pad if needed
    if h < crop_size or w < crop_size:
        pad_h = max(0, crop_size - h)
        pad_w = max(0, crop_size - w)
        data = torch.nn.functional.pad(data, (0, pad_w, 0, pad_h), mode='constant', value=0)
        mask = torch.nn.functional.pad(mask, (0, pad_w, 0, pad_h), mode='constant', value=0)
        h, w = data.shape[2], data.shape[3]
    
    if is_training and h > crop_size and w > crop_size and force_spatial_diversity:
        # Use enhanced crop positioning
        first_mask = mask[0, 0].numpy() if isinstance(mask, torch.Tensor) else mask[0, 0]
        top, left = find_valid_crop_position_enhanced(first_mask, crop_size, force_include_vegetation=True)
    elif h > crop_size and w > crop_size:
        # Random cropping for validation (not center!)
        first_mask = mask[0, 0].numpy() if isinstance(mask, torch.Tensor) else mask[0, 0]
        top, left = find_valid_crop_position_enhanced(first_mask, crop_size, force_include_vegetation=True)
    else:
        # Center crop as last resort
        top = (h - crop_size) // 2
        left = (w - crop_size) // 2
    
    # Apply crop
    data_cropped = data[:, :, top:top+crop_size, left:left+crop_size]
    mask_cropped = mask[:, :, top:top+crop_size, left:left+crop_size]
    
    return data_cropped, mask_cropped

# --- 4. MODEL CLASSES ---
class PhenologyAwareUNet(nn.Module):
    def __init__(self, input_bands=6, hidden_dim=128, output_classes=1):
        super(PhenologyAwareUNet, self).__init__()
        # CNN to extract features from each time step
        self.feature_cnn = nn.Sequential(
            nn.Conv2d(input_bands, 32, kernel_size=3, stride=1, padding=0), nn.ReLU(True), # 32x32 -> 30x30
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=0), nn.ReLU(True), # 30x30 -> 28x28
            nn.MaxPool2d(kernel_size=2, stride=2) # 28x28 -> 14x14
        )
        
        # Feature size after CNN and flattening
        fs = 64 * 14 * 14 # 12544
        
        # LSTM to process the sequence of features
        self.lstm = nn.LSTM(fs, hidden_dim, batch_first=True, bidirectional=True)
        
        # Linear layer to project LSTM output back to feature map size for decoder
        self.projection = nn.Linear(hidden_dim * 2, fs) # hidden_dim * 2 for bidirectional LSTM
        
        # Decoder to reconstruct the segmentation mask
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), # 14x14 -> 14x14
            nn.ReLU(True),
            # Use ConvTranspose2d for proper upsampling
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), # 14x14 -> 28x28
            nn.ReLU(True),
            # Get back to 32x32
            nn.ConvTranspose2d(32, 32, kernel_size=5, stride=1, padding=0), # 28x28 -> 32x32
            nn.ReLU(True),
            nn.Conv2d(32, output_classes, kernel_size=1) # Final output layer
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        
        if h != 32 or w != 32:
            raise ValueError(f"Model expects 32x32 input, got {h}x{w}")
        
        # Process each time step through the CNN
        cnn_outputs = []
        for i in range(t):
            cnn_out = self.feature_cnn(x[:, i])
            cnn_outputs.append(cnn_out.flatten(1))
        
        # Stack CNN outputs for LSTM
        cnn_out = torch.stack(cnn_outputs, 1)
        
        # Pass through LSTM
        lstm_out, _ = self.lstm(cnn_out)

        # Project LSTM output back to CNN feature map size and reshape for decoder
        decoder_outputs = []
        for i in range(t):
            projected = self.projection(lstm_out[:, i]).view(b, 64, 14, 14)
            decoded = self.decoder(projected)
            decoder_outputs.append(decoded)
        
        return torch.stack(decoder_outputs, 1)

# --- 5. SPATIALLY ROBUST DATASET CLASS ---
class SpatiallyRobustVeduDataset(Dataset):
    def __init__(self, data_dir, location_ids, training_window_size=32, is_training=True, 
                 enable_augmentation=True, crops_per_epoch=1):
        self.data_dir = Path(data_dir)
        self.location_ids = [str(gid) for gid in location_ids]
        self.training_window_size = training_window_size
        self.is_training = is_training
        self.enable_augmentation = enable_augmentation
        self.crops_per_epoch = crops_per_epoch
        self.current_epoch = 0  # Initialize current epoch
        
        # Load metadata for each sample
        self.sample_metadata = {}
        for loc_id in self.location_ids:
            metadata = load_sample_metadata(self.data_dir, loc_id)
            self.sample_metadata[loc_id] = metadata
    
    def set_epoch(self, epoch):
        """Set the current epoch for epoch-dependent randomization."""
        self.current_epoch = epoch
    
    def __len__(self): 
        return len(self.location_ids) * self.crops_per_epoch
    
    def __getitem__(self, idx):
        # Add epoch-based randomization to ensure different crops each epoch
        random.seed(idx + self.current_epoch * 42)
        
        # Map linear index to (sample_idx, crop_idx)
        sample_idx = idx // self.crops_per_epoch
        crop_idx = idx % self.crops_per_epoch
        
        loc_id = self.location_ids[sample_idx]
        
        # Load data and target
        data = np.load(self.data_dir / f"{loc_id}_data.npy").astype(np.float32)
        target = np.load(self.data_dir / f"{loc_id}_mask.npy").astype(np.float32)
        
        # Data comes as (T, H, W, C) - transpose to (T, C, H, W)
        data = np.transpose(data, (0, 3, 1, 2))
        
        # Target comes as (T, H, W) - add channel dimension to get (T, 1, H, W)
        if target.ndim == 3:  # (T, H, W)
            target = np.expand_dims(target, axis=1)  # (T, 1, H, W)
        elif target.ndim == 2:  # (H, W) - single time step
            target = np.expand_dims(target, axis=0)  # (1, H, W)
            target = np.expand_dims(target, axis=1)  # (1, 1, H, W)
        
        # Convert to tensors
        data = torch.from_numpy(data)
        target = torch.from_numpy(target)
        
        # Apply enhanced random cropping
        if self.enable_augmentation:
            data, target = enhanced_random_crop_sample(
                data, target, 
                self.training_window_size, 
                is_training=self.is_training,
                force_spatial_diversity=True
            )
        else:
            # Even without augmentation, avoid center cropping
            if data.shape[2] != self.training_window_size or data.shape[3] != self.training_window_size:
                data, target = enhanced_random_crop_sample(
                    data, target, 
                    self.training_window_size, 
                    is_training=False,
                    force_spatial_diversity=False
                )
        
        return data, target

# --- 6. ENHANCED TRAINING FUNCTION ---
def train_spatially_robust_experiment(exp_name, train_ids, val_ids, config, alpha, beta):
    print(f"\n{'='*60}")
    print(f"STARTING SPATIALLY ROBUST EXPERIMENT: {exp_name}")
    print(f"Train samples: {len(train_ids)}, Validation samples: {len(val_ids)}")
    print(f"Alpha: {alpha}, Beta: {beta}")
    print(f"Validation crops per sample: {config['validation_crops_per_sample']}")
    print(f"{'='*60}")
    
    # Training dataset - multiple crops per sample per epoch
    train_dataset = SpatiallyRobustVeduDataset(
        config['data_dir'], train_ids, 
        training_window_size=config['training_window_size'],
        is_training=True,
        enable_augmentation=config['enable_augmentation'],
        crops_per_epoch=3  # 3 different crops per sample per epoch
    )
    
    # Validation dataset - multiple crops per sample for robust validation
    val_dataset = SpatiallyRobustVeduDataset(
        config['data_dir'], val_ids,
        training_window_size=config['training_window_size'],
        is_training=True,  # Still use random cropping for validation!
        enable_augmentation=True,  # Enable augmentation for validation too!
        crops_per_epoch=config['validation_crops_per_sample']
    )
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    
    model = PhenologyAwareUNet(input_bands=config['input_bands'], 
                              hidden_dim=config['hidden_dim'], 
                              output_classes=1).to(config['device'])
    
    criterion = FocalTverskyLoss(alpha=alpha, beta=beta)
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    best_val_loss = float('inf')

    train_losses = []
    val_losses = []

    for epoch in range(config['epochs']):
        # Set epoch for both datasets to ensure different random crops
        train_dataset.set_epoch(epoch)
        val_dataset.set_epoch(epoch)
        
        # Training phase
        model.train()
        epoch_train_loss = 0
        train_batches = 0
        val_losses_per_batch = []
        
        for data, target in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']} Train", leave=False):
            data, target = data.to(config['device']), target.to(config['device'])
            optimizer.zero_grad()
            
            model_output = model(data)
            loss = criterion(model_output.flatten(), target.flatten())
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
            train_batches += 1
        
        avg_train_loss = epoch_train_loss / train_batches if train_batches > 0 else 0
        
        # Validation phase
        model.eval()
        epoch_val_loss = 0
        val_batches = 0
        
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(val_loader):
                data, target = data.to(config['device']), target.to(config['device'])
                model_output = model(data)
                loss = criterion(model_output.flatten(), target.flatten())
                val_losses_per_batch.append(loss.item())
                epoch_val_loss += loss.item()
                val_batches += 1
                
                # Debug: Print first few batches
                if batch_idx < 3 and epoch < 3:  # Only first 3 epochs, first 3 batches
                    print(f"    Val batch {batch_idx}: loss={loss.item():.6f}, "
                          f"pred_range=[{model_output.min():.3f}, {model_output.max():.3f}], "
                          f"target_sum={target.sum().item()}")
        
        avg_val_loss = epoch_val_loss / val_batches if val_batches > 0 else 0
        
        # Print validation loss distribution for first few epochs
        if epoch < 3:
            print(f"    Val loss std: {np.std(val_losses_per_batch):.6f}")
            print(f"    Val loss range: [{min(val_losses_per_batch):.6f}, {max(val_losses_per_batch):.6f}]")
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), config['output_dir'] / f"model_{exp_name}_spatially_robust_best.pt")
        
        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:2d} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    print(f"✅ Spatially robust experiment {exp_name} completed. Best Val Loss: {best_val_loss:.4f}")
    
    # Save training curves
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Spatially Robust Training Curves - {exp_name}')
    plt.legend()
    plt.grid(True)
    plt.savefig(config['output_dir'] / f"spatially_robust_training_curves_{exp_name}.png")
    plt.close()
    
    return {'experiment': exp_name, 'best_val_loss': best_val_loss}

# --- 7. MAIN EXECUTION ---
if __name__ == "__main__":
    # Check that required variables exist
    if 'experimental_sets' not in locals() or 'final_test_set' not in locals():
        raise ValueError("Required variables 'experimental_sets' and 'final_test_set' must be defined.")

    # Get dataset and samples
    best_dataset_name = 'percent_10_area_900'
    print(f"--- Training SPATIALLY ROBUST model on '{best_dataset_name}' ---")

    existing_files = {p.stem.replace("_data", "") for p in CONFIG['data_dir'].glob("*_data.npy")}
    experiment_gdf = experimental_sets[best_dataset_name]
    exp_ids = [gid for gid in experiment_gdf['global_id'].tolist() if gid in existing_files]

    if not exp_ids:
        raise ValueError(f"No existing data files found for dataset '{best_dataset_name}'.")

    # Get stratification data
    stratify_data = experiment_gdf[experiment_gdf['global_id'].isin(exp_ids)]
    stratify_list = stratify_data['is_vedu_present'].tolist()

    print(f"Found {len(exp_ids)} samples")
    print(f"Class distribution: {pd.Series(stratify_list).value_counts().to_dict()}")

    # Create train/val split
    unique_classes, counts = np.unique(stratify_list, return_counts=True)
    min_samples_per_class = min(counts)

    if min_samples_per_class < 2:
        print("Performing random split without stratification")
        exp_train_ids, exp_val_ids = train_test_split(
            exp_ids, test_size=CONFIG['train_val_split_size'], random_state=42
        )
    else:
        print("Performing stratified split")
        exp_train_ids, exp_val_ids = train_test_split(
            exp_ids, test_size=CONFIG['train_val_split_size'], 
            stratify=stratify_list, random_state=42
        )

    print(f"Training set: {len(exp_train_ids)} samples")
    print(f"Validation set: {len(exp_val_ids)} samples")

    # Train spatially robust model
    loss_experiments = [
        {'alpha': 0.5, 'beta': 0.5, 'name': 'balanced_spatially_robust'}
    ]

    for loss_config in loss_experiments:
        result = train_spatially_robust_experiment(
            loss_config['name'], 
            exp_train_ids, 
            exp_val_ids, 
            CONFIG, 
            loss_config['alpha'], 
            loss_config['beta']
        )

    print(f"\n{'='*60}")
    print("✅ SPATIALLY ROBUST TRAINING COMPLETED!")
    print("🧪 Now test this model with the comprehensive testing script")
    print("📊 It should show much better random crop performance!")
    print(f"{'='*60}")

Using device: cuda
Training window size: 32x32
FIXED: Validation will also use random cropping!


ValueError: Required variables 'experimental_sets' and 'final_test_set' must be defined.